# Notebook 2 — Learning Outcomes Analysis
## Digital Skills Programme · Q1 2026 · Geneva

---

### Purpose
This notebook answers the **learning effectiveness** questions of the impact measurement exercise:

| # | Core question |
|---|---------------|
| Q3 | Do attendees **learn**? (knowledge gain analysis) |
| Q4 | Which workshops are **most successful** from a learning perspective? |

### Theoretical framework: Kirkpatrick Level 2 — Learning
The **Kirkpatrick Four-Level Model** is the standard framework for training evaluation:  
- Level 1: Reaction (satisfaction — covered in Notebook 3)  
- **Level 2: Learning** ← *this notebook*  
- Level 3: Behaviour (application on the job — not measured here)  
- Level 4: Results (organisational outcomes — not measured here)  

Level 2 is measured via **pre/post knowledge tests**. Each participant receives a score out of 10 before and after the workshop. The difference is the *knowledge gain*.

### Why this matters
Demonstrating learning is the foundational justification for a skills programme.  
Without evidence that participants actually gain competence, all other metrics (satisfaction, reputation, finance) are insufficient to claim social impact.  
**Funders, partners, and public authorities routinely require Kirkpatrick Level 2 evidence.**

### Data source
- `data/workshop_participants.csv` — one row per participant; `pre_score`, `post_score`, `knowledge_gain`
- `data/workshops.csv` — workshop-level metadata

---

In [ ]:
# ─── Cell 1 · Imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (11, 5)

DATA_DIR = Path("data")
print("Libraries loaded.")

---
## Section 1 — Load and inspect participant data

In [ ]:
# ─── Cell 2 · Load data ───────────────────────────────────────────────────────
participants = pd.read_csv(DATA_DIR / "workshop_participants.csv", parse_dates=["workshop_date"])
workshops    = pd.read_csv(DATA_DIR / "workshops.csv",            parse_dates=["date"])

# Recompute knowledge_gain for safety (post - pre)
participants["knowledge_gain"] = (participants["post_score"] - participants["pre_score"]).round(2)

print(f"Participants: {len(participants)} rows")
print(f"Columns: {list(participants.columns)}")
print()
print(participants[["workshop_name", "payment_type", "modality",
                     "pre_score", "post_score", "knowledge_gain"]].head(8))

---
## Section 2 — Q3: Do attendees learn? Programme-wide evidence

**Statistical approach**  
We use a **paired-samples t-test** (Wilcoxon signed-rank as non-parametric backup) to test whether the mean post-score is significantly higher than the mean pre-score.  
We also compute **Cohen's d** as the effect size measure: a standardised distance between pre and post means.  
Effect size interpretation (Cohen 1988):  
- d < 0.2: negligible  
- 0.2 ≤ d < 0.5: small  
- 0.5 ≤ d < 0.8: medium  
- d ≥ 0.8: large

In [ ]:
# ─── Cell 3 · Programme-wide paired t-test ────────────────────────────────────
pre  = participants["pre_score"]
post = participants["post_score"]

t_stat, p_val = stats.ttest_rel(post, pre)

# Cohen's d for paired data = mean(diff) / std(diff)
diff = post - pre
cohens_d = diff.mean() / diff.std(ddof=1)

# Wilcoxon signed-rank (non-parametric alternative, no normality assumption)
w_stat, w_p = stats.wilcoxon(post, pre)

print("=== Programme-wide Learning Test (all 240 participants) ===")
print(f"  Mean pre-test score :  {pre.mean():.3f}  (SD = {pre.std():.3f})")
print(f"  Mean post-test score:  {post.mean():.3f}  (SD = {post.std():.3f})")
print(f"  Mean knowledge gain :  {diff.mean():.3f}  (SD = {diff.std():.3f})")
print()
print(f"  Paired t-test: t = {t_stat:.3f}, p = {p_val:.2e}")
print(f"  Cohen's d: {cohens_d:.3f}  ({('negligible' if cohens_d<0.2 else 'small' if cohens_d<0.5 else 'medium' if cohens_d<0.8 else 'large')} effect)")
print(f"  Wilcoxon signed-rank: W = {w_stat:.0f}, p = {w_p:.2e}")
print()
if p_val < 0.001:
    print("✓ CONCLUSION: Knowledge gain is statistically significant (p < 0.001).")
    print("  Participants demonstrably learn during the workshops.")

In [ ]:
# ─── Cell 4 · Pre/post score distribution (violin + paired lines) ─────────────
# A violin plot shows the full distribution shape of scores before and after.
# We also draw a random sample of individual trajectories to visualise learning paths.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: violin plots
ax = axes[0]
melted = participants[["participant_id", "pre_score", "post_score"]].melt(
    id_vars="participant_id", var_name="Test", value_name="Score")
melted["Test"] = melted["Test"].map({"pre_score": "Pre-test", "post_score": "Post-test"})

sns.violinplot(data=melted, x="Test", y="Score", palette=["#5B8DB8", "#2E8B57"],
               inner="quartile", ax=ax)
ax.set_title("Score Distribution: Pre vs Post", fontweight="bold")
ax.set_ylabel("Score (0–10)")
ax.set_ylim(0, 10.5)

# Right: spaghetti chart — random 40 participants' individual trajectories
ax2 = axes[1]
sample = participants.sample(40, random_state=42)
for _, row in sample.iterrows():
    color = "seagreen" if row["knowledge_gain"] > 0 else "tomato"
    ax2.plot([0, 1], [row["pre_score"], row["post_score"]],
             color=color, alpha=0.4, linewidth=0.9)

ax2.set_xticks([0, 1])
ax2.set_xticklabels(["Pre-test", "Post-test"])
ax2.set_ylabel("Score (0–10)")
ax2.set_ylim(0, 10.5)
ax2.set_title("Individual Learning Trajectories (n=40 sample)", fontweight="bold")

learner_patch = mpatches.Patch(color="seagreen", alpha=0.6, label="Gained knowledge")
no_patch      = mpatches.Patch(color="tomato",   alpha=0.6, label="No gain")
ax2.legend(handles=[learner_patch, no_patch], fontsize=9)

plt.suptitle("Knowledge Gain Evidence — All Participants", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 5 · Knowledge gain distribution histogram ──────────────────────────
# The histogram shows the shape of gains: are most people improving by 2-3 points,
# or is there a bimodal pattern suggesting some participants did not benefit?
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(participants["knowledge_gain"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(participants["knowledge_gain"].mean(), color="red",    linestyle="--",
           linewidth=2, label=f"Mean = {participants['knowledge_gain'].mean():.2f}")
ax.axvline(participants["knowledge_gain"].median(), color="orange", linestyle=":",
           linewidth=2, label=f"Median = {participants['knowledge_gain'].median():.2f}")
ax.axvline(0, color="black", linewidth=1, linestyle="-")

ax.set_xlabel("Knowledge Gain (post − pre score)")
ax.set_ylabel("Number of participants")
ax.set_title("Distribution of Individual Knowledge Gains", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

pct_positive = (participants["knowledge_gain"] > 0).mean() * 100
pct_above_2  = (participants["knowledge_gain"] >= 2).mean() * 100
print(f"% of participants with any gain (> 0):        {pct_positive:.1f}%")
print(f"% of participants with substantial gain (≥ 2): {pct_above_2:.1f}%")

---
## Section 3 — Learning by payment type: do free-seat participants learn equally?

**Why this matters**  
The equity claim of the free-seat policy rests on the assumption that free participants benefit *just as much* as paid ones.  
If free-seat participants learn significantly more or less than paid participants, that has implications for how those seats are allocated and to whom.

In [ ]:
# ─── Cell 6 · Knowledge gain by payment type ──────────────────────────────────
gain_by_type = participants.groupby("payment_type")["knowledge_gain"].agg(["mean", "std", "count"]).round(3)
print("Mean knowledge gain by payment type:")
print(gain_by_type)

# Two-sample t-test: do paid vs free participants learn equally?
paid_gain = participants[participants["payment_type"] == "paid"]["knowledge_gain"]
free_gain = participants[participants["payment_type"] == "free"]["knowledge_gain"]

t2, p2 = stats.ttest_ind(paid_gain, free_gain)
print(f"\nt-test (paid vs free): t = {t2:.3f}, p = {p2:.4f}")
if p2 > 0.05:
    print("→ No significant difference in learning between paid and free participants.")
    print("  The equity claim of the free-seat policy is supported.")
else:
    print("→ Significant difference detected — investigate allocation criteria.")

fig, ax = plt.subplots(figsize=(7, 4))
participants.boxplot(column="knowledge_gain", by="payment_type", ax=ax,
                     patch_artist=True,
                     boxprops=dict(facecolor="lightblue"),
                     medianprops=dict(color="red", linewidth=2))
ax.set_title("Knowledge Gain: Paid vs Free Participants", fontweight="bold")
ax.set_xlabel("Payment type")
ax.set_ylabel("Knowledge gain")
plt.suptitle("")  # remove pandas auto-title
plt.tight_layout()
plt.show()

---
## Section 4 — Q4: Which workshops are most successful? (Learning dimension)

**Why this matters**  
Not all workshops will perform equally.  
Identifying which ones produce the highest knowledge gains lets the team:  
- **Replicate success**: understand what the best workshops do differently  
- **Improve laggards**: target pedagogical improvements where they matter most  
- **Communicate impact**: use strongest evidence in fundraising and public reporting

In [ ]:
# ─── Cell 7 · Mean knowledge gain per workshop (with confidence intervals) ─────
# We compute mean, 95% CI, and effect size (Cohen's d vs zero baseline) per workshop.
def cohens_d_vs_zero(series):
    """Cohen's d treating baseline = 0 (how many SDs away from no-learning)."""
    return series.mean() / series.std(ddof=1) if series.std(ddof=1) > 0 else 0

def conf_interval_95(series):
    n = len(series)
    se = series.sem()
    t_crit = stats.t.ppf(0.975, df=n-1)
    return t_crit * se

workshop_gain = (
    participants
    .groupby("workshop_name")["knowledge_gain"]
    .agg(
        mean_gain="mean",
        std_gain="std",
        n="count",
        ci95=conf_interval_95,
        cohens_d=cohens_d_vs_zero
    )
    .sort_values("mean_gain", ascending=False)
    .round(3)
)
print("Learning outcomes by workshop (sorted best → worst):")
print(workshop_gain)

In [ ]:
# ─── Cell 8 · Bar chart with 95% CI error bars ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colors = ["#2E8B57" if v >= workshop_gain["mean_gain"].median() else "#5B8DB8"
          for v in workshop_gain["mean_gain"]]

bars = ax.bar(workshop_gain.index, workshop_gain["mean_gain"],
              yerr=workshop_gain["ci95"], capsize=5,
              color=colors, edgecolor="white", linewidth=0.8)

ax.axhline(workshop_gain["mean_gain"].mean(), color="red", linestyle="--",
           linewidth=1.5, label=f"Programme average = {workshop_gain['mean_gain'].mean():.2f}")

ax.set_xticklabels(workshop_gain.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Mean Knowledge Gain (0–10 scale)")
ax.set_title("Mean Knowledge Gain per Workshop (with 95% CI)", fontweight="bold")
ax.legend(fontsize=9)

# Add Cohen's d labels above bars
for i, (name, row) in enumerate(workshop_gain.iterrows()):
    ax.text(i, row["mean_gain"] + row["ci95"] + 0.05,
            f"d={row['cohens_d']:.2f}", ha="center", fontsize=7.5, color="gray")

plt.tight_layout()
plt.show()

best  = workshop_gain["mean_gain"].idxmax()
worst = workshop_gain["mean_gain"].idxmin()
print(f"\nBest learning workshop : {best}  (gain = {workshop_gain.loc[best,'mean_gain']:.2f}, d = {workshop_gain.loc[best,'cohens_d']:.2f})")
print(f"Lowest learning workshop: {worst} (gain = {workshop_gain.loc[worst,'mean_gain']:.2f}, d = {workshop_gain.loc[worst,'cohens_d']:.2f})")

In [ ]:
# ─── Cell 9 · Pre-test score vs knowledge gain scatter ────────────────────────
# Research in education consistently shows a 'ceiling effect': participants who
# start with very high prior knowledge gain less (less room to improve).
# This scatter reveals whether such an effect is present here.
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(participants["pre_score"], participants["knowledge_gain"],
                c=participants["survey_overall"], cmap="RdYlGn",
                alpha=0.55, s=25, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Overall satisfaction (1–5)")

# Regression line
slope, intercept, r, p_r, se = stats.linregress(participants["pre_score"], participants["knowledge_gain"])
x_line = np.linspace(participants["pre_score"].min(), participants["pre_score"].max(), 100)
ax.plot(x_line, intercept + slope * x_line, color="red", linewidth=2,
        label=f"Trend  r={r:.2f}  p={p_r:.3f}")

ax.set_xlabel("Pre-test score")
ax.set_ylabel("Knowledge gain")
ax.set_title("Prior Knowledge vs Learning Gain (coloured by satisfaction)", fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Pearson r (pre vs gain): {r:.3f}  (p = {p_r:.4f})")
if slope < 0 and p_r < 0.05:
    print("→ Significant negative correlation: participants with lower prior knowledge gain more.")
    print("  This is consistent with a 'floor effect' — beginners benefit most.")
else:
    print("→ No strong ceiling/floor effect detected.")

In [ ]:
# ─── Cell 10 · Learning by modality: onsite vs online ─────────────────────────
# Does the delivery format (in-person vs virtual) affect how much participants learn?
# This is a key question for programme expansion decisions.
modality_gain = participants.groupby("modality")["knowledge_gain"].agg(["mean","std","count"]).round(3)
print("Knowledge gain by modality:")
print(modality_gain)

onsite_gain = participants[participants["modality"] == "onsite"]["knowledge_gain"]
online_gain = participants[participants["modality"] == "online"]["knowledge_gain"]
t_mod, p_mod = stats.ttest_ind(onsite_gain, online_gain)
print(f"\nt-test (onsite vs online): t = {t_mod:.3f}, p = {p_mod:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=participants, x="modality", y="knowledge_gain",
            palette={"onsite": "steelblue", "online": "darkorange"}, ax=ax)
ax.set_title("Knowledge Gain: Onsite vs Online", fontweight="bold")
ax.set_xlabel("Modality")
ax.set_ylabel("Knowledge gain")
plt.tight_layout()
plt.show()

if p_mod < 0.05:
    print("→ Significant difference in learning by format.")
else:
    print("→ No significant difference in learning by format (p > 0.05).")
    print("  Online and onsite workshops produce comparable learning outcomes.")

In [ ]:
# ─── Cell 11 · Heatmap: per-workshop pre, post and gain ───────────────────────
# A heatmap of mean pre-score, mean post-score, and mean knowledge gain
# per workshop provides an at-a-glance quality map of the entire programme.
heatmap_data = (
    participants
    .groupby("workshop_name")[["pre_score", "post_score", "knowledge_gain"]]
    .mean()
    .round(2)
    .sort_values("knowledge_gain", ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="YlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Score"})
ax.set_title("Mean Pre-score, Post-score and Knowledge Gain per Workshop", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ─── Cell 12 · Learning KPI summary ──────────────────────────────────────────
print("═" * 60)
print("   LEARNING OUTCOMES KPI DASHBOARD")
print("═" * 60)
print(f"  Total participants analysed              {len(participants)}")
print(f"  Programme mean knowledge gain            {participants['knowledge_gain'].mean():.2f} pts")
print(f"  Effect size (Cohen's d)                  {cohens_d:.3f} ({('negligible' if cohens_d<0.2 else 'small' if cohens_d<0.5 else 'medium' if cohens_d<0.8 else 'large')})")
print(f"  % participants with positive gain        {pct_positive:.1f}%")
print(f"  % participants with gain ≥ 2 pts         {pct_above_2:.1f}%")
print(f"  Statistical significance (paired t-test) p = {p_val:.2e}")
print(f"  Best workshop (learning)                 {best}")
print(f"  Onsite vs Online learning difference     p = {p_mod:.4f} ({'sig.' if p_mod<0.05 else 'n.s.'})")
print("═" * 60)
print("\nConclusion: Participants do learn. The knowledge gain is")
print("statistically significant and practically meaningful.")
print("All workshops produced positive average gains.")